<a href="https://colab.research.google.com/github/Safeenaa07/Food-delivery-website/blob/main/Predictmedinstruments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

# -------------------------------------------------------------
# 1. Create Synthetic Medical Dataset
# -------------------------------------------------------------
np.random.seed(42)
num_samples = 1500

# Features (medical indicators)
df = pd.DataFrame({
    "age": np.random.randint(20, 80, num_samples),
    "bmi": np.round(np.random.uniform(18, 40, num_samples), 2),
    "heart_rate": np.random.randint(55, 110, num_samples),
    "insulin_level": np.random.randint(5, 300, num_samples),
    "activity_level": np.random.randint(1, 10, num_samples)  # 1–10 scale
})

# Target: Glucose Level (continuous)
df["glucose_level"] = (
    df["age"] * 0.3 +
    df["bmi"] * 1.8 +
    df["heart_rate"] * 0.5 +
    df["insulin_level"] * 0.7 -
    df["activity_level"] * 1.2 +
    np.random.randint(-15, 15, num_samples)  # noise
)

print("\nSample Dataset:")
print(df.head())

# -------------------------------------------------------------
# 2. Train/Test Split
# -------------------------------------------------------------
X = df.drop("glucose_level", axis=1)
y = df["glucose_level"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -------------------------------------------------------------
# 3. Feature Scaling
# -------------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------------------------------------------
# 4. Deep Learning Regression Model
# -------------------------------------------------------------
model = keras.Sequential([
    layers.Dense(64, activation="relu", input_shape=(5,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1)  # output = continuous value
])

model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()

# -------------------------------------------------------------
# 5. Train Model
# -------------------------------------------------------------
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    verbose=1
)

# -------------------------------------------------------------
# 6. Evaluate Model
# -------------------------------------------------------------
loss, mae = model.evaluate(X_test_scaled, y_test)
print(f"\nTest MAE: {mae:.2f}")
print(f"Test MSE: {loss:.2f}")

# -------------------------------------------------------------
# 7. Predict Example Patient’s Glucose Level
# -------------------------------------------------------------
sample_patient = pd.DataFrame([{
    "age": 45,
    "bmi": 26.4,
    "heart_rate": 78,
    "insulin_level": 130,
    "activity_level": 5
}])

sample_scaled = scaler.transform(sample_patient)
pred = model.predict(sample_scaled)

print("\nPredicted Glucose Level:", float(pred[0][0]))



Sample Dataset:
   age    bmi  heart_rate  insulin_level  activity_level  glucose_level
0   58  21.64         109            287               6        315.552
1   71  22.10          95            238               3        285.580
2   48  36.42          93             27               9        136.556
3   34  25.31          67            157               1        192.958
4   62  24.85          65            281               3        301.930


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 64)             │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,657 (26.00 KB)

 Trainable params: 6,657 (26.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 6s 25ms/step - loss: 48793.4219 - mae: 211.9619 - val_loss: 45151.7734 - val_mae: 203.9772
Epoch 2/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 46608.2266 - mae: 206.8930 - val_loss: 41164.9570 - val_mae: 194.0799
Epoch 3/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 39557.2344 - mae: 189.9363 - val_loss: 26417.2773 - val_mae: 152.5176
Epoch 4/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 21913.3633 - mae: 135.2763 - val_loss: 4649.7324 - val_mae: 57.5304
Epoch 5/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 2908.8787 - mae: 43.6001 - val_loss: 1456.4976 - val_mae: 29.8172
Epoch 6/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 1476.7858 - mae: 30.6672 - val_loss: 976.7228 - val_mae: 25.5490
Epoch 7/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 1053.5348 - mae: 25.9531 - val_loss: 894.2626 - val_mae: 24.1867
Epoch 8/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 922.6290 - mae: 24.1508 - val_loss: 844.6263 - val_mae